# Module 3: Machine Learning
In Module 1, we showed some of the easiest ways to understand AI with regression and classification lines.  Module 2, we transition from trying to draw a bunch of lines into Neural Networks. In this module, I wanted to introduce you to concept of Machine Learning. This particular expression of AI is trying to learn from data without being explicitly with every class or every rule. 

In this setup; let us assume we are using the Predictor that uses the Neural Network from Module 2.
![equation_2_4.png](attachment:bc9b5a73-4319-4670-8bea-994e3e532e02.png)

As we feed the Neural Network real data, it is possible that this code:
  print("\nClass probabilities:")
  print(f" Bream : {probs_user[0] * 100:.1f}%")
  print(f" Roach : {probs_user[1] * 100:.1f}%")
  print(f" Pike  : {probs_user[2] * 100:.1f}%")

could be used by a Machine Learning system downstream to determine if we want to keep the fish or not.  

The real-world rule is: "We will only keep the fish if we can definitively identify what it is."  In electronics, this is called an Exclusive OR; or XOR. 




In [ ]:
# Imports

import numpy as np

# Each row is a training example, each column is a feature

X=np.array((
[0, 0, 0],
[0, 0, 1],
[0, 1, 0],
[1, 0, 0],
[1, 0, 1],
[1, 1, 1]), dtype=float)

y=np.array((
[0],
[1],
[1],
[1],
[0],
[0]), dtype=float)

# ============================================================

# Define useful functions

# ============================================================

# Activation function

def sigmoid(t):
    return 1/(1+np.exp(-t))

# Derivative of sigmoid

def sigmoid_derivative(p):
    return p * (1 - p)

# ============================================================

# Neural Network Class

# ============================================================

class NeuralNetwork:


    def __init__(self, x, y):
        
        # Initialize the Neural Network input to x
        self.input = x
        
        # Initialize the computed output set to be the same size as y
        self.y = y
        self.output = np.zeros(self.y.shape)
        
        # Define the size and hidden layers of the neural network
        self.lr = 0.9
        self.inputlayer_neurons = self.input.shape[1]
        self.hiddenlayer_neurons = 4
        self.hiddenlayer2_neurons = 5
        self.output_neurons = 1
        
        # Initialize the weights
        self.wh = np.random.uniform(
            size=(self.inputlayer_neurons, self.hiddenlayer_neurons)
        )
        
        self.wh2 = np.random.uniform(
            size=(self.hiddenlayer_neurons, self.hiddenlayer2_neurons)
        )
        
        self.wout = np.random.uniform(
            size=(self.hiddenlayer2_neurons, self.output_neurons)
        )
        
        # Initialize activation layer storage
        self.hiddenlayer_activations = np.random.uniform(
            size=(self.hiddenlayer_neurons, self.output.shape[1])
        )
        
        self.hiddenlayer2_activations = np.random.uniform(
            size=(self.hiddenlayer_neurons, self.hiddenlayer2_neurons)
        )
        
        
    # ========================================================
    # Feedforward
    # ========================================================
    
    def feedforward(self):
        
        hidden_layer_input = np.dot(self.input, self.wh)
        
        self.hiddenlayer_activations = sigmoid(hidden_layer_input)
        
        hiddenlayer2_input = np.dot(
            self.hiddenlayer_activations,
            self.wh2
        )
        
        self.hiddenlayer2_activations = sigmoid(hiddenlayer2_input)
        
        output_layer_input = np.dot(
            self.hiddenlayer2_activations,
            self.wout
        )
        
        return sigmoid(output_layer_input)
        
        
    # ========================================================
    # Backpropagation
    # ========================================================
    
    def backprop(self):
        
        # Calculate the slopes
        slope_output_layer = sigmoid_derivative(self.output)
        
        slope_hidden_layer2 = sigmoid_derivative(
            self.hiddenlayer2_activations
        )
        
        slope_hidden_layer = sigmoid_derivative(
            self.hiddenlayer_activations
        )
        
        # Calculate errors
        E = self.y - self.output
        
        d_output = E * slope_output_layer
        
        Error_at_hidden_layer2 = d_output.dot(self.wout.T)
        
        d_hiddenlayer2 = (
            Error_at_hidden_layer2 *
            slope_hidden_layer2
        )
        
        Error_at_hidden_layer = (
            Error_at_hidden_layer2.dot(self.wh2.T)
        )
        
        d_hiddenlayer = (
            Error_at_hidden_layer *
            slope_hidden_layer
        )
        
        # Update the weights
        self.wout += (
            self.hiddenlayer2_activations.T.dot(d_output)
            * self.lr
        )
        
        self.wh2 += (
            self.hiddenlayer_activations.T.dot(d_hiddenlayer2)
            * self.lr
        )
        
        self.wh += (
            self.input.T.dot(d_hiddenlayer)
            * self.lr
        )
        
        
    # ========================================================
    # Train
    # ========================================================
    
    def train(self, X, y):
        
        self.output = self.feedforward()
        self.backprop()
        
        
    # ========================================================
    # Predict
    # ========================================================
    
    def predict(self, X):
        
        self.input = X
        
        return self.feedforward()

# ============================================================
# Create Neural Network
# ============================================================

NN = NeuralNetwork(X, y)

# ============================================================
# MACHINE LEARNING: BEST WEIGHT TRACKING
# We could have just relied on letting the NN run and see where
# it ends. But with incomplete data, small data sets, we can
# not assume the last Epoch will necessarily be the best Epoch.
# ============================================================
# Start with an infinitely high loss.
# The first calculated loss will automatically become
# the best loss.

best_loss = float("inf")

# These will store the weights associated with best_loss.

best_wh = None
best_wh2 = None
best_wout = None

# ============================================================
# Train the Neural Network
# ============================================================

for i in range(1000):

    # Train the network
    NN.train(X, y)
    
    # Calculate the current loss AFTER the weight update
    predictions = NN.feedforward()
    current_loss = np.mean(np.square(y - predictions))


# --------------------------------------------------------
# MACHINE LEARNING REWARD:
# The best weights are the ones with the smallest loss
# This code check whether this is the best model so far
# --------------------------------------------------------

if current_loss < best_loss:
    
    # Save the new best loss
    best_loss = current_loss
    
    # IMPORTANT:
    # Make copies of the weights.
    # We use .copy() so later training does not overwrite
    # our saved best weights.
    best_wh = NN.wh.copy()
    best_wh2 = NN.wh2.copy()
    best_wout = NN.wout.copy()


# --------------------------------------------------------
# Display progress every 500 iterations
# --------------------------------------------------------

if i % 50 == 0:
    
    print("for iteration # " + str(i) + "\n")
    print("Input : \n" + str(X))
    print("Actual Output: \n" + str(y))
    print("Predicted Output: \n" + str(predictions))
    print("Loss: \n" + str(current_loss))
    print("Best Loss So Far: \n" + str(best_loss))
    print("\n")

# ============================================================
# RESTORE THE BEST WEIGHTS
# ============================================================
# The network has now completed training.
# Replace the final weights with the weights that produced
# the LOWEST training loss.

NN.wh = best_wh.copy()
NN.wh2 = best_wh2.copy()
NN.wout = best_wout.copy()

# ============================================================

# Final Training Results Using BEST WEIGHTS

# ============================================================

final_training_predictions = NN.predict(X)

final_training_loss = np.mean(
np.square(y - final_training_predictions)
)

print("\n========================================")
print("BEST MODEL")
print("========================================")
print("Lowest Training Loss:")
print(final_training_loss)

print("\nPredictions Using Best Weights:")
print(final_training_predictions)

# ============================================================
# Final Prediction / Test Data
# ============================================================

X2 = np.array((
[0,0,0],
[0,0,1],
[0,1,0],
[0,1,1],
[1,0,0],
[1,0,1],
[1,1,0],
[1,1,1]
), dtype=float)

y2 = np.array((
[0],
[1],
[1],
[0],
[1],
[0],
[0],
[0]
), dtype=float)

# Predict using the BEST weights
predictions = NN.predict(X2)

# Calculate test loss
test_loss = np.mean(
np.square(y2 - predictions)
)

# ============================================================
# Display Final Prediction
# ============================================================
print("\n========================================")
print("FINAL TEST USING BEST MODEL")
print("========================================")
print("Input : \n", X2)
print("Expected Output: \n", y2)
print("Predicted Output: \n", predictions)
print("Test Loss: \n", test_loss)
print("\nLowest Training Loss: ", best_loss)